# 📊 Sales Intelligence AI — Professional EDA
### Unlocking Marketing ROI & Revenue Drivers through Exploratory Data Analysis
*Created by Principal Data Scientist & UI/UX Designer*

---

## 📌 Executive Summary
This notebook performs a comprehensive **Exploratory Data Analysis (EDA)** on the advertising dataset (`Advertising.csv`). The objective is to identify how marketing spend across three distinct channels—**TV**, **Radio**, and **Newspaper**—impacts product **Sales**.

Our goal is to extract mathematical insights that will drive the **Sales Intelligence AI** machine learning models and budget optimization algorithms.

---

## 📁 1. Data Loading & Setup
We load the dataset and configure Plotly to render premium dark mode visualizations.

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
import plotly.io as pio

# Set default template to match our premium dark theme
pio.templates.default = 'plotly_dark'

print('Libraries imported successfully!')

In [ ]:
df = pd.read_csv('Advertising.csv')

# Remove the index column if it exists
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print(f'Dataset Shape: {df.shape}')
df.head()

## 🔍 2. Data Quality & Integrity Report
We examine the dataset for data types, missing values, duplicates, and outlier anomalies.

In [ ]:
print('--- Data Types & Non-Null Counts ---')
df.info()

print('\n--- Missing Values ---')
print(df.isnull().sum())

print('\n--- Duplicates Check ---')
print(f'Total Duplicate Rows: {df.duplicated().sum()}')

### 📦 Outlier Analysis (Box & Violin Plots)
We use a combination of Plotly **Box Plots** and **Violin Plots** to identify structural anomalies, check for skewness, and visualize density.

In [ ]:
fig_box = px.box(df, y=['TV', 'Radio', 'Newspaper'],
                 title='Advertising Spend Distribution across Channels',
                 labels={'value': 'Spend (in $1,000s)', 'variable': 'Channel'},
                 color_discrete_sequence=['#4F8CFF', '#7C3AED', '#22C55E'])
fig_box.update_layout(
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(11, 18, 32, 0.7)',
    font_color='#E2E8F0'
)
fig_box.show()

> 💡 **Business Insights (Box Plots):**
> * **TV advertising** has the largest budget range (up to $300k) with no outliers. It shows a fairly uniform distribution, indicating experiments were run across the entire spend spectrum.
> * **Radio advertising** has a narrower spend range ($0 to $50k) with a symmetric distribution and no outliers.
> * **Newspaper advertising** has a right-skewed distribution with a couple of high-spend outliers exceeding the upper fence (~$93.6k). These represent campaign budgets that are disproportionately high compared to typical spending levels.

In [ ]:
fig_violin = px.violin(df, y='Sales', box=True, points='all',
                       title='Target Variable (Sales) Distribution & Density',
                       labels={'Sales': 'Sales (in 1,000 units)'},
                       color_discrete_sequence=['#EF4444'])
fig_violin.update_layout(
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(11, 18, 32, 0.7)',
    font_color='#E2E8F0'
)
fig_violin.show()

> 💡 **Business Insights (Violin Plots):**
> * The **Sales** target variable exhibits a unimodal, slightly right-skewed distribution.
> * Most campaigns yield sales between 10k and 17k units, peaking around 12.9k.
> * The distribution lacks outliers, indicating sales numbers are stable and suitable for robust regression modeling.

## 📈 3. Univariate Distribution Analysis
Let's dive deeper into the individual distributions of our advertising channels using histograms to understand distribution shapes.

In [ ]:
# Create interactive histograms
for col, color in zip(['TV', 'Radio', 'Newspaper'], ['#4F8CFF', '#7C3AED', '#22C55E']):
    fig_hist = px.histogram(df, x=col, marginal='rug',
                            title=f'Distribution of {col} Advertising Spend',
                            labels={col: f'{col} Spend ($1,000s)'},
                            color_discrete_sequence=[color])
    fig_hist.update_layout(
        plot_bgcolor='rgba(0,0,0,0)',
        paper_bgcolor='rgba(11, 18, 32, 0.7)',
        font_color='#E2E8F0'
)
    fig_hist.show()

> 💡 **Business Insights (Univariate Distributions):**
> * **TV & Radio Spend** are almost uniformly distributed. This is characteristic of controlled experimental datasets designed to evaluate predictive relationships without systemic bias.
> * **Newspaper Spend** is heavily clustered near the lower end ($0 to $20k), with a long tail stretching to $114k. This represents a traditional 'diminishing returns' layout, where large spends are rare, likely due to low perceived performance.

## 🔗 4. Bivariate Analysis: Spend vs. Sales
To evaluate how spend in each channel influences sales directly, we create scatter plots with fitted Ordinary Least Squares (OLS) regression lines.

In [ ]:
# Scatter plots with OLS trendlines
for col, color in zip(['TV', 'Radio', 'Newspaper'], ['#4F8CFF', '#7C3AED', '#22C55E']):
    fig_scatter = px.scatter(df, x=col, y='Sales', trendline='ols',
                             title=f'{col} Spend vs. Sales Relationship',
                             labels={col: f'{col} Spend ($1,000s)', 'Sales': 'Sales (1,000 units)'},
                             color_discrete_sequence=[color],
                             trendline_color_override='#EF4444')
    fig_scatter.update_layout(
        plot_bgcolor='rgba(0,0,0,0)',
        paper_bgcolor='rgba(11, 18, 32, 0.7)',
        font_color='#E2E8F0'
    )
    fig_scatter.show()

> 💡 **Business Insights (Bivariate Analysis):**
> * **TV vs. Sales**: Strong, highly linear relationship ($R \approx 0.78$). The tight clustering of data points around the regression line indicates a predictable, reliable return on investment for TV campaigns.
> * **Radio vs. Sales**: Strong positive relationship ($R \approx 0.58$), but with higher dispersion. This indicates that while Radio is effective, sales outputs are more volatile and might depend on other variables (e.g., interaction effects).
> * **Newspaper vs. Sales**: Weak linear relationship ($R \approx 0.23$). The points are highly scattered, and the flat slope of the regression line indicates that Newspaper spend has a minor effect on sales.

## 📊 5. Multivariate Analysis & Feature Interactions
Let's analyze correlation matrices and pairplots to understand how advertising budgets interact and identify potential multicollinearity.

In [ ]:
corr_matrix = df.corr()

fig_heat = ff.create_annotated_heatmap(
    z=corr_matrix.values,
    x=list(corr_matrix.columns),
    y=list(corr_matrix.index),
    colorscale='Viridis',
    annotation_text=corr_matrix.round(3).values
)
fig_heat.update_layout(
    title='Feature Correlation Matrix Heatmap',
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(11, 18, 32, 0.7)',
    font_color='#E2E8F0'
)
fig_heat.show()

> 💡 **Business Insights (Correlation Heatmap):**
> * **TV has the highest correlation with Sales (0.782)**, followed by **Radio (0.576)**. These are your primary growth levers.
> * **Newspaper has a very low correlation (0.228)**, suggesting it is a less efficient channel.
> * **Radio and Newspaper have a correlation of 0.354**, indicating a moderate interaction. This could suggest they were frequently bundled together in campaigns, which can introduce mild multicollinearity in simple models.

In [ ]:
fig_matrix = px.scatter_matrix(df,
                               dimensions=['TV', 'Radio', 'Newspaper', 'Sales'],
                               title='Scatter Plot Pair Matrix (Pairplot/Scatter Matrix)',
                               color_discrete_sequence=['#7C3AED'])
fig_matrix.update_layout(
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(11, 18, 32, 0.7)',
    font_color='#E2E8F0'
)
# Make points slightly transparent for density visibility
fig_matrix.update_traces(diagonal_visible=False, marker=dict(opacity=0.6, size=5))
fig_matrix.show()

> 💡 **Business Insights (Pairplot Analysis):**
> * The scatter matrix visualizes all bivariate combinations simultaneously.
> * Notice the non-linear 'wedge' shape in the TV vs. Sales and Radio vs. Sales relationships. At very low budgets, sales are low, but the combination of TV and Radio spend has a multiplicative effect (synergy), which a standard linear model might underestimate without interaction terms.

## 💡 6. Strategic Marketing Recommendations & Modeling Strategy

### Strategic Channel Guidance
1. **Prioritize TV as the Foundation**: TV spend is highly correlated with sales. High-budget campaigns should focus on TV as it provides a solid base rate of returns.
2. **Leverage Radio for Synergy**: Radio has a strong positive slope. However, because of its lower saturation point (max budget ~$50k in the dataset), it should be paired with TV campaigns.
3. **Minimize Newspaper Budgets**: Newspaper spend shows weak predictive capacity and high dispersion. Budgets in this channel should be audited and re-allocated to TV and Radio, except for specialized target segments.

### Modeling Strategy
*   **Linear Regression with Interaction Terms**: Since TV and Radio show strong linear trends but potential synergy, including an interaction term ($TV \times Radio$) in the regression model will likely improve predictive accuracy ($R^2$).
*   **Tree-based Models**: A Random Forest or Gradient Boosting model should be trained alongside linear regression to capture non-linear relationships and check feature importance.